# 02 · Behavioural evaluation
Look at receiver responses and metrics side by side for paired conditions.

In [ ]:
from pathlib import Path
import json
import pandas as pd

RESULTS = Path("../results")
# Pick a run explicitly, or use the most recent one that has trials.parquet.
RUN_ID = None
runs = sorted(p for p in RESULTS.iterdir() if (p / "trials.parquet").exists())
RUN = RESULTS / RUN_ID if RUN_ID else runs[-1]
print(RUN.name)

In [ ]:
df = pd.read_parquet(RUN / "trials.parquet")
print(df.status.value_counts())
metrics = ["metrics.response_words", "metrics.task_success", "metrics.marker_present",
           "metrics.resp_vs_source_contradiction_rate", "metrics.response_sentiment_negative"]
df.groupby(["condition"])[[m for m in metrics if m in df]].mean()

In [ ]:
# One pair: same source, task, receiver and seed, differing only by condition.
pair = df[df.status == "ok"].pair_key.iloc[0]
for _, row in df[df.pair_key == pair].sort_values("condition").iterrows():
    r = json.loads((RUN / row.response_artifact).read_text())
    print(f"--- {row.condition} | {row.task} | {row.receiver} | wm={row.watermark_enabled} | p={row.watermark_p_value:.3g}")
    print("SYSTEM:", r["request"]["messages"][0]["content"][-200:])
    print("ANSWER:", r["text"][:500], "\n")

In [ ]:
# Per-task success by condition
df[df.status == "ok"].pivot_table(index="task", columns="condition", values="metrics.task_success", aggfunc="mean")